# धडा १८: क्रिप्टोग्राफिक रिसीट्ससह AI एजंट्सची सुरक्षा

## हाताने करण्यायोग्य नोटबुक

ही नोटबुक चार व्यायामांमधून मार्गदर्शन करते:

१. एजन्ट टूल कॉलसाठी **तुमची पहिली रिसीट साइन करा** आणि त्याची पडताळणी करा.
२. रिसीटमध्ये **छेडछाड करा** आणि पडताळणी अपयशी होऊन बघा.
३. **तीन-रिसीट चेन तयार करा** आणि चेनची अखंडता पुष्टी करा.
४. **मायक्रोसॉफ्ट एजंट फ्रेमवर्क टूल कॉलवर रॅप करा** जेणेकरून प्रत्येक क्रिया रिसीट जाहीर करेल.

सर्व क्रिप्टोग्राफिक क्रिया चांगल्या प्रकारे देखभाल केलेल्या लायब्ररींपासून आयात केल्या आहेत (`pynacl` Ed25519 साठी, `jcs` RFC 8785 कॅनॉनिकल JSON साठी, `hashlib` SHA-256 साठी Python मानक लायब्ररीमधून). रिसीट लॉजिक स्वतः साधे Python आहे जे तुम्ही वाचू आणि सुधारू शकता.

कक्षां प्रमाणे चालवा. प्रत्येक विभाग लहान आणि स्व-समावेशक आहे.


## सेटअप

दोन अवलंबन स्थापित करा. दोघांकडे मोकळ्या परवाना (Apache-2.0 / MIT) आहेत.


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## सहाय्यक युटिलिटीज

हे दोन सहाय्यक बेस64url एनकोडिंग (पॅडिंग शिवाय) आणि मनमोकळ्या वस्तूंचे SHA-256 हॅशिंग हाताळतात. ते नोटबुकचा उर्वरित भाग रसीद लॉजिकवर लक्ष केंद्रित ठेवतात.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## विभाग 1: आपल्या पहिल्या पावतीवर स्वाक्षरी करा

कल्पना करा की आमचा एजंट **Contoso Travel** साठी नुकताच सिडनी ते लॉस एंजेलिसतील फ्लाइट्स ग्राहकासाठी शोधले आहेत. आम्हाला या टूल कॉल विषयी गेटवेचे विधान नोंदवायचे आहे जेणेकरून भविष्यात एखादा ऑडिटर याची वेगळ्या पिन केलेल्या जारी करणाऱ्या कीखाली पडताळणी करू शकेल.

### पाऊल 1.1: स्वाक्षरी की तयार करा

उत्पादनात, एजंटची स्वाक्षरी की हार्डवेअर सुरक्षा मॉड्यूल (HSM), Azure Key Vault, किंवा तत्सम संरक्षित संग्रहस्थानात राहील. या धड्यासाठी आम्ही मेमरीमध्ये नवीन की तयार करतो.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### टप्पा 1.2: रसीद पेलोड तयार करा

पेलोडमध्ये त्यातील सर्व काही असते जे आम्हाला रसीदने प्रमाणित करायचे आहे: कोण कार्यरत होता, कोणते साधन वापरले, कोणत्या अर्ग्युमेंटसह, काय परत आले, कोणत्या धोरणाअंतर्गत, आणि केव्हा. आम्ही अर्ग्युमेंट्स आणि निकाल यांचा हॅश करतो, त्यांना थेट समाविष्ट करण्याऐवजी, जेणेकरून रसीद संवेदनशील सामग्री लीक होणार नाही.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### टप्पा 1.3: पावतीवर सही करा आणि संकलित करा

तीन टप्पे:

1. JCS वापरून पेलोडचे कॅनॉनिकलायझेशन करा जेणेकरून दोन अंमलबजावण्या सारखा तार्किक पावती तयार केल्यास त्याच बाइट्स तयार होतील.
2. Ed25519 खाजगी कीने थेट कॅनॉनिकल बाइट्सवर सही करा. PureEdDSA संदेशाचा अंतर्गत हॅश करतो, त्यामुळे अतिरिक्त प्री-हॅश केल्यास प्रोटोकॉल बदलेल.

नंतर सही मूळ पेलोडशी जोडली जाते ज्यामुळे अंतिम पावती तयार केली जाते.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### पाऊल 1.4: पावतीची पडताळणी करा

पडताळणी प्रक्रियेचा उलटा आहे. आम्ही सही काढून टाकतो, कॅनॉनिकल बाइट्स पुन्हा गणना करतो, आणि सही जारीकर्त्याच्या सार्वजनिक कीशी तपासतो.

पडताळणी ऑफलाइन असते: कॉल करण्यासाठी कोणतीही सेवा नाही आणि कोणताही नेटवर्क प्रवेश नाही. तरीही याला एक गोष्ट पावतीव्यतिरिक्त आवश्यक आहे: जारीकर्त्याची सार्वजनिक की, जी ऑडिटर आधीच विश्वास ठेवतो अशा चॅनेलद्वारे मिळते, जसे की जारीकर्ता प्रकाशित केलेला JWK सेट किंवा पूर्वनिर्धारित पिन केलेली किंमत. पावतीतील `public_key` फील्ड ही भूमिका बजावू शकत नाही, कारण जो कोणी पावती लिहितो तोच ते फील्ड लिहितो. विभाग 2 हे टाळण्यास मदत करणारी बनावट दाखवतो.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


तुम्हाला `Receipt is valid: True`, `Pre-hashed receipt valid: False` आणि `Malformed receipts rejected: True` दिसायला हव्यात. सकारात्मक केसमुळे direct-JCS स्वाक्षरी मार्ग कार्यान्वीत होतो हे सिद्ध होते; नकारात्मक नियंत्रणामुळे स्वाक्षरी-व्याप्ती नियम चालवण्यायोग्य बनतो, त्याऐवजी फक्त मजकूर म्हणून राहत नाही; शेवटच्या ओळीत खराब स्वरूपातील इनपुटमुळे अपवाद न उभा राहता `False` परत येतो हे दिसते.


## विभाग 2: रिसीटमध्ये फेरफार करा

रिसीटचा पूर्ण उद्देश म्हणजे ती फेरफार-प्रमाणित असावी. चला हे सिद्ध करूया.

आम्ही रिसीटचा अगदी एकच अक्षर बदलू आणि पडताळणी अयशस्वी होते का पाहू.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### काय घडलं?

जेव्हा आपण `policy_id` बदललं, तेव्हा canonical bytes बदलले. सही (जी मूळ canonical bytes वर होती) आता जुळत नाही. पडताळणी योग्यरित्या `False` परत करते.

स्वाक्षरी केलेल्या क्षेत्रात, म्हणजे `signature` ऑब्जेक्टच्या बाहेरील काहीही बदलण्याचा वरील प्रमाणपत्रावर सत्यापन असणे शक्य नाही, जोपर्यंत हल्लेखोरांकडे जारीकर्त्याची खाजगी कळी नाही. `signature` ऑब्जेक्ट स्वतः स्वाक्षरी केलेला नाही: `sig` ही सही आहे, आणि `verify_receipt` `public_key` दुर्लक्षित करतो, त्यामुळे त्या फील्डची फक्त संपादन करणे काहीही बदलत नाही. जोपर्यंत खाजगी कळी कीष्ठकात आहे आणि सत्यापनकर्ते जारीकर्त्याची स्वतंत्रपणे प्रकाशित केलेली कळी वापरतात, तसपर्यंत स्वाक्षरी केलेल्या फील्डमध्ये बदल लपविणे अशक्य आहे. ही अट महत्त्वाची आहे: पुढील सेलमध्ये एक बदललेले प्राप्ती दाखवले आहे ज्याची पडताळणी वेगळी कळी वापरून परिपूर्णपणे होते.

स्वतः प्रयत्न करा: वरील सेलमधील `tool_name` किंवा `agent_id` किंवा `timestamp` बदलून पुन्हा चालवा. स्वाक्षरी केलेल्या क्षेत्रातील प्रत्येक बदल अयोग्य प्राप्ती तयार करतो.


### सह्या जे टॅम्पर पकडत नाही

वरचा हल्लेखोर पुनः स्वाक्षरी करत नाही. एक अधिक काळजीपूर्वक व्यक्ती करेल: रिसिप्टमध्ये बदल करेल,
स्वतः तयार केलेल्या कीने त्यावर सही करेल, आणि त्या कीला
`public_key` क्षेत्रात ठेवेल. परिणामी, रिसिप्टची सही तिच्या सामग्रीसाठी पूर्णपणे खरी असेल.


हे चालवा आणि पाहा कोणती दोन तपासण्यांपैकी कोणती पकडते.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


तुम्हाला `True` नंतर `False` दिसायला हवे. ती तीन `assert` ओळी ती एक
तपासणी करतात: जेव्हा खोटेपणा कधीही जारीकर्त्याच्या कीशी पडताळणी करतो तेव्हा सेल त्रुटी निर्माण करतो.


खोटेपणा अंतर्गत सुसंगत आहे. त्याचा सही खरोखर त्याच्या
सामग्रीशी आणि त्याने वाहलेली कीशी जुळतो, त्यामुळे की रसीदेतून वाचणारा पडताळणी करणारा
तो वैध असल्याची नोंद करतो. रसीदेमध्ये काहीही चुकीचे दिसत नाही, कारण
त्यामध्ये काहीही विसंगत नाही.

याला पकडणारी एकमेव गोष्ट म्हणजे अशी कीशी तपासणी करणे ज्यावर हल्लेखोराकडे
प्रभाव टाकण्यासाठी मार्ग. म्हणूनच `verify_receipt` कीला एक आर्ग्युमेंट म्हणून घेतो: एक
ज्याची सिग्नेचर `verify_receipt(receipt)` अशी फंक्शन सुरक्षितपणे लिहिली जाऊ शकत नाही,
कारण त्याला मिळणारी एकटी की तीच आहे जी हॅकरने दिली असते.

तोच रसीद `sample_receipts/04_resigned_receipt.json` म्हणून सेव्ह केली जाते जर तुम्हाला
ती नोटबुकबाहेर तपासायची असेल.


## विभाग 3: पावत्या एकत्र जोडा

एकच पावती एका क्रियेचे संरक्षण करते. बहुतेक एजंट अनेक क्रिया करतात. संपूर्ण अनुक्रमणिका छेडछाड-प्रमाणित बनविण्यासाठी, आम्ही प्रत्येक पावतीला मागील पावतीच्या हॅशला नवीन पावतीच्या पेलोडमध्ये समाविष्ट करून मागील पावतीशी लिंक करतो.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

जर कोणी सादर केलेल्या अनुक्रमणिकेमधील एखादी पावती काढली किंवा पुनर्रचने केली, तर पुढील लिंक तुटते. छाटणी किंवा सहीदार नियंत्रित उपसंग्रह पुन्हा लिहिणे शोधण्यासाठी सध्याचा पिन केलेला हेड, बाह्य चेकपॉइंट किंवा प्रमाणित लोकसंख्या प्रतिबद्धता आवश्यक असते.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


आता मधल्या पावत्या बिघडवून साखळी तोडा आणि पुन्हा खात्री करा. बिघडवलेली पावती तिच्या सही तपासणीत अयशस्वी होते, आणि पुढील पावती तिच्या साखळी-लिंक तपासणीत अयशस्वी होते (कारण तिचा `previous_receipt_hash` आता बदललेल्या मधल्या पावतीच्या हॅशशी जुळत नाही).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


रसीद 0 अजूनही सत्यापित होते (ते सुधारलेले नाही आणि त्यावर अवलंबून राहण्यासाठी कोणताही पूर्वगामी नाही). रसीद 1 त्याच्या स्वाक्षरी तपासणीमध्ये अयशस्वी होते कारण आपण `tool_args_hash` बदलले आहे. रसीद 2 त्याच्या साखळी-संदर्भ तपासणीत अयशस्वी होते कारण त्याचा `previous_receipt_hash` मूळ (आता सुधारित) रसीद 1 च्या विरोधात गणना केला गेला होता.

जरी हल्लेखोराने सुधारित रसीद 1 ला पुन्हा स्वाक्षरी केली तरी नवीन स्वाक्षरी जारीकर्त्याच्या की विरुद्ध अयशस्वी होते जर ते जारीकर्त्याची खाजगी की न पकडत असतील (त्यांच्या स्वतःच्या कीने पुन्हा स्वाक्षरी करणे अयशस्वी होते, जसे की विभाग 2 मध्ये दाखवलेले आहे), आणि रसीद 2 मधील साखळी-संदर्भ विसंगती अजूनही छेडछाड उघड करेल. बदल लपवण्यासाठी, हल्लेखोराने सुधारणा बिंदूपासून पुढील प्रत्येक रसीद पुन्हा स्वाक्षरी करावी लागेल, ज्यासाठी जारीकर्त्याची खाजगी की हवी असते.


## विभाग 4: रिसीट स्वाक्षरीसह एजंट टूल कॉल लपेटा

वास्तविक तैनातीत, तुम्हाला प्रत्येक एजंट लेखकाकडून `make_receipt` कॉल करण्याची स्मरणशक्ती हवयं लागत नाही. प्रत्येक टूल कॉलसाठी रिसीट स्वाक्षरी स्वयंचलित असावी.

येथे सर्वात सोपा नमुना आहे: एक wrapper वर्ग जो कोणत्याही callable टूल फंक्शन्स घेतो आणि त्याचा रिसीट-प्रशिक्षित आवृत्ती परत करतो. हे कोणत्याही एजंट फ्रेमवर्कला, Microsoft Agent Framework (`agent_framework.foundry`) सह, अनुकूल बनतो.

जर तुमच्याकडे Microsoft Foundry प्रकल्प सेट केलेला नसेल, तरीही खालील स्थानिक mock हा नमुना दाखवतो.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### मायक्रोसॉफ्ट एजंट फ्रेमवर्कसह एकत्रिकरण

वरील `ReceiptedTool` रॅपर फ्रेमवर्क-स्वतंत्र आहे. मायक्रोसॉफ्ट एजंट फ्रेमवर्कने तयार केलेल्या एजंटमध्ये ते वापरण्यासाठी, रॅप केलेल्या फंक्शनला टूल म्हणून नोंदणी करा. एक रेखाचित्र (आपण मॉकला खऱ्या मायक्रोसॉफ्ट फाउंड्री टूल नोंदणीने बदलाल):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

एजंट फ्रेमवर्कला रिसीप्ट्सबद्दल काहीही माहित असण्याची गरज नाही. रिसीप्ट साइनिंग टूलच्या भोवती रॅप केली जाते, फ्रेमवर्कमध्ये नको अशा प्रकारे जोडली जात नाही. एजंटला पुन्हा लिहिले नाही तर विद्यमान एजंट कोडमध्ये असा प्रॉव्हिनन्स कसा जोडायचा हे आहे. 


## पुनरावलोकन आणि विस्तार आव्हान

तुमच्याकडे आहे:

- Ed25519 की जोडी तयार केली आहे.
- एजंट टूल कॉलसाठी रिसीप्ट तयार केली आणि सही केली आहे.
- फक्त जारी करणाऱ्याच्या सार्वजनिक कीचा वापर करून रिसीप्ट ऑफलाइन सत्यापित केली आहे, जी रिसीप्टपासून स्वतंत्रपणे मिळवली गेली आहे.
- रिसीप्टमध्ये फेरफार केला आणि सत्यापनाला अपयश मिळाले हे पाहिले.
- तीन रिसीप्ट्सची हॅश-चेन केलेली क्रमवार रचना तयार केली.
- साखळीत मधल्या भागात फेरफार केला आणि सही अपयशी आणि साखळी लिंक अपयशी यांचे निरीक्षण केले.
- एका टूल फंक्शनला स्वयंचलित रिसीप्ट सहीसह गुंडाळले.

**विस्तार आव्हान.** रिसीप्ट स्कीममध्ये `request_id` क्षेत्र (वितरित ट्रेसिंगसाठी UUID) जोडा. `make_receipt` मध्ये ते समाविष्ट करा, आणि पुष्टी करा की रिसीप्ट्स अखेरपर्यंत सत्यापित होतात. त्यानंतर सही केल्यानंतर हे क्षेत्र बदला आणि सत्यापन अपयशी होते याची पुष्टी करा. यामुळे canonical encoding च्या प्रत्येक बाइटचा सहीकडे कसा उपयोग होतो हे तुम्हाला समजेल.

**महत्त्वाची मर्यादा.** एका निश्चित की आणि साखळी संदर्भाखाली, रिसीप्ट्स सहीचे स्रोत, सामग्री अखंडता आणि सादर केलेल्या पूर्वीच्या लिंकचा पुरावा आहेत. ते अंमलबजावणी, स्रोत सत्य, विश्वासार्ह वेळ, पूर्णता, धोरण मूल्यांकन किंवा शुद्धता सिद्ध करीत नाहीत. रिसीप्ट्स एक पाया आहे. शासन तुमच्या तयार केलेल्या प्रणालीवर आहे.

त्या मर्यादेसह धडा README पुन्हा वाचा. रिसीप्ट्स बद्दल संघ जो सर्वसाधारण चुका करतात ती म्हणजे "आपल्याकडे रिसीप्ट्स आहेत" म्हणजे "आपण शासित आहोत" असं गृहित धरणं. तसे नाही. रिसीप्ट्स एजंटच्या वर्तनाला तपासण्याजोगे करतात. ते योग्य करीत नाहीत.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
